# Filter ethene dynamic CASSCF data

In [5]:
# Discard a geometry when its C=C bond is longer than 2.0 Å 
# or any of its four C--H bonds is longer than 1.5 Å. 
# Complete extended-XYZ records, including valid CASSCF labels, are retained unchanged.

from pathlib import Path

import numpy as np

INPUT_XYZ = Path("../data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF.xyz")
OUTPUT_XYZ = Path("../data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF_filtered.xyz")
MAX_CC_BOND = 2.0  # Å
MAX_CH_BOND = 1.5  # Å

# Atom order in this dataset is C, C, H, H, H, H.
CC_PAIR = (0, 1)
CH_PAIRS = ((0, 2), (0, 5), (1, 3), (1, 4))


def distance(positions, atom_pair):
    first, second = atom_pair
    return float(np.linalg.norm(positions[first] - positions[second]))


stats = {"total": 0, "kept": 0, "discarded_cc": 0, "discarded_ch": 0, "discarded_both": 0}
with INPUT_XYZ.open() as source, OUTPUT_XYZ.open("w") as destination:
    while atom_count_line := source.readline():
        atom_count = int(atom_count_line)
        comment_line = source.readline()
        atom_lines = [source.readline() for _ in range(atom_count)]
        if not comment_line or len(atom_lines) != atom_count or any(not line for line in atom_lines):
            raise ValueError(f"Incomplete frame after frame {stats['total']}")

        atom_fields = [line.split() for line in atom_lines]
        symbols = [fields[0] for fields in atom_fields]
        if atom_count != 6 or symbols != ["C", "C", "H", "H", "H", "H"]:
            raise ValueError(f"Unexpected atom ordering in frame {stats['total']}: {symbols}")
        positions = np.array([[float(value) for value in fields[1:4]] for fields in atom_fields])

        cc_bond = distance(positions, CC_PAIR)
        longest_ch_bond = max(distance(positions, pair) for pair in CH_PAIRS)
        exceeds_cc = cc_bond > MAX_CC_BOND
        exceeds_ch = longest_ch_bond > MAX_CH_BOND

        stats["total"] += 1
        if exceeds_cc:
            stats["discarded_cc"] += 1
        if exceeds_ch:
            stats["discarded_ch"] += 1
        if exceeds_cc and exceeds_ch:
            stats["discarded_both"] += 1
        if exceeds_cc or exceeds_ch:
            continue

        destination.write(atom_count_line)
        destination.write(comment_line)
        destination.writelines(atom_lines)
        stats["kept"] += 1

stats["discarded"] = stats["total"] - stats["kept"]
print(f"Wrote {stats['kept']} retained geometries to {OUTPUT_XYZ.resolve()}")
print(stats)


Wrote 35003 retained geometries to /home/lim_yt/X-MACE-sampling/data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF_filtered.xyz
{'total': 62031, 'kept': 35003, 'discarded_cc': 8642, 'discarded_ch': 18701, 'discarded_both': 315, 'discarded': 27028}


In [ ]:
# Filter data using shnitsel tools sanity_check 
# defaults: any bond 3A, CH or NH bond 2A
# active state potential step 0.7, hop potential step 1.0, total energy step 0.1, total energy drift 0.2, kinetic energy step 0.7

from pathlib import Path
import numpy as np
import shnitsel as st
from shnitsel.clean import sanity_check
from shnitsel.clean.filter_geo import GeometryFiltrationThresholds

dt = st.io.read("../data/A01_ethene/dynamic/A01_ethene_0p50fs_dynamic.nc")
# The input stores Cartesian coordinates in angstrom, but omits the metadata
# required by shnitsel when it constructs the molecule from `atXYZ`.
# so set units manually
dt.dataset["atXYZ"].attrs["units"] = "angstrom"

truncated_dt = sanity_check(
    dt, "truncate"
)
print(f"Retained {truncated_dt.dataset.sizes['frame']} frames")

/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/shnitsel/units/conversion.py:235: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'frame'} to avoid an error in the future.
  tmp = data.assign(new_vars)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/shnitsel/clean/filter_energy.py:171: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'frame'} to avoid an error in the future.
  res["epot_hop_step"] = mdiff(e_pot_active).where(is_hop, 0)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/site-packages/shnitsel/clean/filter_energy.py:183: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'frame'} to avoid

Retained 38641 frames


In [8]:
# Filter extended-XYZ CASSCF records with the applicable defaults from
# shnitsel.clean.sanity_check. The source lacks trajectory, active-state, and
# kinetic-energy metadata, so this checks geometry plus all REF_energy states.

import json
import re
from pathlib import Path

import numpy as np

INPUT_XYZ = Path("../data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF.xyz")
OUTPUT_XYZ = Path(
    "../data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF_sanity_defaults_filtered.xyz"
)
MAX_BOND_LENGTH = 3.0  # Å: sanity_check default for every bond
MAX_CH_BOND_LENGTH = 2.0  # Å: sanity_check default for C--H/N--H bonds
MAX_ENERGY_STEP = 0.7  # eV: applied to every REF_energy state
CC_PAIR = (0, 1)
CH_PAIRS = ((0, 2), (0, 5), (1, 3), (1, 4))
ENERGY_PATTERN = re.compile(r'REF_energy="_JSON (\[\[.*?\]\])"')


def read_xyz_record(source, record_index):
    atom_count_line = source.readline()
    if not atom_count_line:
        return None
    try:
        atom_count = int(atom_count_line)
    except ValueError as error:
        raise ValueError(f"Invalid atom count in record {record_index}") from error

    comment_line = source.readline()
    atom_lines = [source.readline() for _ in range(atom_count)]
    if not comment_line or any(not line for line in atom_lines):
        raise ValueError(f"Incomplete record {record_index}")
    return atom_count_line, comment_line, atom_lines


def parse_record(record, record_index):
    atom_count_line, comment_line, atom_lines = record
    fields = [line.split() for line in atom_lines]
    if any(len(parts) < 4 for parts in fields):
        raise ValueError(f"Invalid atom line in record {record_index}")
    symbols = [parts[0] for parts in fields]
    if len(fields) != 6 or symbols != ["C", "C", "H", "H", "H", "H"]:
        raise ValueError(f"Unexpected atom ordering in record {record_index}: {symbols}")
    try:
        positions = np.array([[float(value) for value in parts[1:4]] for parts in fields])
    except (IndexError, ValueError) as error:
        raise ValueError(f"Invalid coordinates in record {record_index}") from error

    match = ENERGY_PATTERN.search(comment_line)
    if match is None:
        raise ValueError(f"Missing REF_energy in record {record_index}")
    try:
        energies = np.asarray(json.loads(match.group(1)), dtype=float).squeeze()
    except (json.JSONDecodeError, ValueError) as error:
        raise ValueError(f"Invalid REF_energy in record {record_index}") from error
    if energies.shape != (3,):
        raise ValueError(f"Expected three REF_energy states in record {record_index}")
    return positions, energies


def bond_length(positions, atom_pair):
    first, second = atom_pair
    return float(np.linalg.norm(positions[first] - positions[second]))


stats = {
    "total": 0, "kept": 0, "rejected_geometry": 0,
    "rejected_energy": 0, "rejected_both": 0,
}
previous_energies = None
with INPUT_XYZ.open() as source, OUTPUT_XYZ.open("w") as destination:
    while (record := read_xyz_record(source, stats["total"])) is not None:
        positions, energies = parse_record(record, stats["total"])
        cc_bond = bond_length(positions, CC_PAIR)
        longest_ch_bond = max(bond_length(positions, pair) for pair in CH_PAIRS)
        fails_geometry = (
            cc_bond > MAX_BOND_LENGTH
            or longest_ch_bond > MAX_BOND_LENGTH
            or longest_ch_bond > MAX_CH_BOND_LENGTH
        )
        fails_energy = (
            previous_energies is not None
            and np.any(np.abs(energies - previous_energies) > MAX_ENERGY_STEP)
        )

        stats["total"] += 1
        if fails_geometry:
            stats["rejected_geometry"] += 1
        if fails_energy:
            stats["rejected_energy"] += 1
        if fails_geometry and fails_energy:
            stats["rejected_both"] += 1
        if not (fails_geometry or fails_energy):
            atom_count_line, comment_line, atom_lines = record
            destination.write(atom_count_line)
            destination.write(comment_line)
            destination.writelines(atom_lines)
            stats["kept"] += 1

        previous_energies = energies

stats["rejected"] = stats["total"] - stats["kept"]
print(f"Wrote {stats['kept']} retained records to {OUTPUT_XYZ.resolve()}")
print(stats)


Wrote 39791 retained records to /home/lim_yt/X-MACE-sampling/data/A01_ethene/dynamic/A01_ethene_dynamic_CASSCF_sanity_defaults_filtered.xyz
{'total': 62031, 'kept': 39791, 'rejected_geometry': 20602, 'rejected_energy': 2106, 'rejected_both': 468, 'rejected': 22240}
